In [1]:
%pip install -q openai google-genai anthropic python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

for key in ["OPENAI_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"]:
    print(f"{key}: {'SET' if os.getenv(key) else 'MISSING'}")

OPENAI_API_KEY: SET
GEMINI_API_KEY: SET
ANTHROPIC_API_KEY: SET


In [3]:
OPENAI_MODEL = "gpt-4.1-nano"
GEMINI_MODEL = "gemini-3.1-flash-lite"
ANTHROPIC_MODEL = "claude-haiku-4.5"

In [4]:
from google import genai

client_gemini = genai.Client()

# interaction = clinet_gemini.models.generate_content(  
#     model=GEMINI_MODEL,
#     input="Say something about Bangladesh in one sentence.",
# )
# print(interaction.text)

interaction = client_gemini.interactions.create(
    model=GEMINI_MODEL,
    input="Say something about Bangladesh in one sentence.",
)

print(interaction.output_text)

Bangladesh is a vibrant South Asian nation known for its lush riverine landscapes, resilient culture, and rapidly growing economy.


In [5]:
from openai import OpenAI
client_openai = OpenAI()

response = client_openai.responses.create(
    model=OPENAI_MODEL,
    input="Say something about Bangladesh in one sentence.",
)

print(response.output_text)

Bangladesh is a vibrant South Asian nation known for its rich cultural heritage, lush landscapes, and the tranquil beauty of the Sundarbans mangrove forest.


In [ ]:
import anthropic

client_anthropic = anthropic.Anthropic()

message = client_anthropic.messages.create(
  model=ANTHROPIC_MODEL,
  max_tokens=1024,
  messages=[{
    "role": "user",
    "content": "Say something about Bangladesh in one sentence.",
  }],
)
print(message.content[0].text)

In [6]:
SYSTEM = "You are a support assistant for Northstar Services. Answer in ONE sentence."
USER   = "A customer asks how to reset their password."

In [8]:
# Gemini

# from google.genai import types
# interaction = clinet_gemini.models.generate_content(
#     model=GEMINI_MODEL,
#     contents=USER,
#     config=types.GenerateContentConfig(system_instruction=SYSTEM),
# )
# print(interaction.text)

interaction = client_gemini.interactions.create(
    model=GEMINI_MODEL,
    system_instruction=SYSTEM,
    input=USER,
    # input="Say something about Bangladesh in one sentence.",
)

print("Gemini : ", interaction.output_text)

Gemini :  To reset your password, please visit the Northstar Services login page and click the "Forgot Password" link to receive a secure recovery email.


In [9]:
# OpenAI
response = client_openai.responses.create(
    model=OPENAI_MODEL,
    instructions=SYSTEM,
    input=USER,
    # input="Say something about Bangladesh in one sentence.", (Previous)

)

print("OpenAI : ", response.output_text)

OpenAI :  Please visit the password reset page on our website and follow the instructions to reset your password.


In [ ]:
# Anthropic
message = client_anthropic.messages.create(
  model=ANTHROPIC_MODEL,
  max_tokens=1024,
  system=SYSTEM,
  messages=[{
    "role": "user",
    "content": USER,
  }],
)
print("Anthropic : ", message.content[0].text)

In [10]:
USER = "I was charged twice this month. Can I get a refund?"
def ask_gemini(system=None):
    """Send USER to Gemini with an optional system prompt; return just the reply text."""
    kwargs = dict(
        model="gemini-2.5-flash",  
        input=USER,
    )
    if system:
        kwargs["system_instruction"] = system
        
    response = client_gemini.interactions.create(**kwargs)
    return response.output_text

In [11]:
print(ask_gemini())

Absolutely, dealing with a double charge can be frustrating, but **yes, you can almost always get a refund for a duplicate charge.**

Here's how you should proceed:

1.  **Gather All Your Information:**
    *   **Who charged you?** (The name of the company or merchant)
    *   **What was the charge for?** (Product, service, subscription, etc.)
    *   **Exact Dates of Both Charges:** Make sure they are identical transactions.
    *   **Exact Amounts of Both Charges:** Confirm they are the same.
    *   **Transaction Numbers or Order IDs:** If available on your statement or receipt.
    *   **Proof:** Screenshots of your bank/credit card statement showing both charges, original receipts, or order confirmations.

2.  **Contact the Company/Merchant Directly First:**
    *   This is the quickest and most common way to resolve a double charge.
    *   **Find their customer service contact information:** Look on their website for a "Contact Us" page, support email, phone number, or live chat

In [12]:
SYSTEM = "You are a friendly customer-support agent for Northstar Services. Be warm, clear, and concise."

print(ask_gemini(SYSTEM))

Oh dear, I'm so sorry to hear you've been charged twice this month! We'll definitely get that sorted out for you.

To investigate this and process your refund, I'll need to securely access your account details. Could you please provide your Northstar Services account number or the email address associated with your account?

Once I have that, I can look into the duplicate charge right away.


In [13]:
SYSTEM = (
    "You are a friendly customer-support agent for Northstar Services. Be warm, clear, and concise.\n"
    "Reply in no more than TWO sentences."        
)

print(ask_gemini(SYSTEM))

Oh no, I'm sorry to hear about the double charge! Please provide your account number or the email associated with your service so I can look into this and process your refund for you.


In [14]:
SYSTEM = (
    "You are a support agent for Northstar Services, a SaaS company.\n"
    "- Be friendly, calm, and concise (2-3 sentences).\n"
    "- NEVER invent account details, charges, or refund decisions.\n"          # <-- guardrail
    "- Refunds and billing disputes MUST be escalated to a human - say so.\n"  # <-- guardrail
    "- If you are unsure, offer to connect them with a teammate rather than guess."  # <-- fallback
)

print(ask_gemini(SYSTEM))

I understand you were charged twice this month and are requesting a refund. Billing inquiries and refunds need to be handled by our dedicated support team. I'd be happy to connect you with a teammate who can investigate this further and assist you.


In [15]:
chat = client_gemini.chats.create(model=GEMINI_MODEL)
print("Gemini 1 :", chat.send_message("My invoice looks wrong.").text)
print("Gemini 2 :", chat.send_message("It's INV-4821.").text)  

Gemini 1 : I can certainly help you figure this out. Since I don’t have access to your private accounts or billing systems, I need you to act as my "eyes" on the document.

To help me identify the problem, **please tell me a bit more about what you are seeing.** 

Here are the most common things to check. Does one of these apply?

### 1. Is there a math error?
*   Did they charge you for more items than you received?
*   Is the tax rate incorrect?
*   Are the subtotal and the final total not adding up?

### 2. Is there a pricing discrepancy?
*   Did they charge you a higher price than what was quoted or agreed upon in your contract?
*   Are you being charged for a service or subscription you already cancelled?
*   Are there "hidden" fees that weren't disclosed?

### 3. Is there a credit issue?
*   Did you make a payment that isn't showing up as a credit?
*   Did you return an item that wasn't deducted from the final amount?

---

### How to proceed:

1.  **If you are comfortable sharin

In [17]:
from pydantic import BaseModel, Field
from typing import Literal

MESSAGE = "I've been charged twice for May and nobody has replied to my emails. This is ridiculous."

def extractGemini(schema):
    """Ask Gemini to fill in `schema` from MESSAGE; return the validated object."""
    response = client_gemini.interactions.create(
        model=GEMINI_MODEL,
        input=MESSAGE,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": schema.model_json_schema()
        },
    )
    output = schema.model_validate_json(response.output_text)
    return output

In [18]:
def extractOpenAI(schema):
    """Ask OpenAI to fill in `schema` from MESSAGE; return the validated object."""
    response = client_openai.responses.parse(
        model=OPENAI_MODEL,
        input=MESSAGE,
        text_format=schema
    )
    return response.output_parsed

In [ ]:
def extractAntropic(schema):
    """Ask Antropic to fill in `schema` from MESSAGE; return the validated object."""
    response = client_anthropic.messages.parse(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": MESSAGE}],
        output_format=schema
    )
    return response.parsed_output

In [19]:
class TriageV0(BaseModel):
    summary: str          

r1 = extractOpenAI(TriageV0)
# r2 = extractAntropic(TriageV0)
r3 = extractGemini(TriageV0)
print("OpenAI Structured Response:\n", r1)
# print("Antropic Structured Response:\n", r2)
print("Gemini Structured Response:\n", r3)

OpenAI Structured Response:
 summary="I'm sorry to hear about the double charge and the lack of response. I recommend contacting your bank or credit card company to dispute the extra charge and continue following up with the company's customer service, possibly through phone if emails are not responded to. If you'd like, I can help craft a follow-up email or suggest additional steps."
Gemini Structured Response:
 summary='Customer reported being double-charged for the month of May and is expressing frustration over a lack of response to their emails.'


In [22]:
class TriageV1(BaseModel):
    category: str         
    summary:  str

t1 = extractGemini(TriageV1)
print(t1)
print("Unpredictable value ->", repr(t1.category))   

t2 = extractOpenAI(TriageV1)
print(t2)
print("Unpredictable value ->", repr(t2.category))  

category='Billing/Payment Issue' summary='Customer reports a double charge for the month of May and states that previous email inquiries have gone unanswered.'
Unpredictable value -> 'Billing/Payment Issue'
category='Billing Issue' summary='You have been double-charged for May and have not received responses to your emails. Consider contacting customer support through alternative methods or requesting a supervisor to resolve the issue promptly.'
Unpredictable value -> 'Billing Issue'


In [24]:
class TriageV2(BaseModel):
    category: Literal["billing", "technical", "account", "general"]   # a fixed menu
    summary:  str

t1 = extractGemini(TriageV2)
print(t1)
print("Guaranteed one of the four ->", t1.category)

t2 = extractOpenAI(TriageV2)
print(t2)
print("Guaranteed one of the four ->", t2.category)

category='billing' summary='Customer reported a double charge for May and stated that previous email inquiries have gone unanswered.'
Guaranteed one of the four -> billing
category='billing' summary='Duplicate charges for May without response to emails.'
Guaranteed one of the four -> billing


In [25]:
class Triage(BaseModel):
    category:  Literal["billing", "technical", "account", "general"]
    urgency:   Literal["low", "medium", "high"]
    sentiment: Literal["negative", "neutral", "positive"]
    summary:   str = Field(description="One-line summary of what the customer wants")  

t1 = extractGemini(Triage)
print(t1)
print("\nSwitch on any field directly ->", t1.category, "|", t1.urgency, "|", t1.sentiment)

t2 = extractOpenAI(Triage)
print(t2)
print("\nSwitch on any field directly ->", t2.category, "|", t2.urgency, "|", t2.sentiment)

category='billing' urgency='high' sentiment='negative' summary='Customer is requesting a refund for a duplicate charge in May after receiving no response to previous emails.'

Switch on any field directly -> billing | high | negative
category='billing' urgency='high' sentiment='negative' summary='Customer was double charged for May and has not received a reply to emails.'

Switch on any field directly -> billing | high | negative


In [28]:
import openai
from google import genai
from google.genai import errors

oai = openai.OpenAI(max_retries=3, timeout=30)
gem = genai.Client()


try:
    resp_openai = oai.responses.create(model=OPENAI_MODEL, input="hi")
    print("OpenAI:", resp_openai.output_text)
except openai.RateLimitError:
    print("OpenAI Error: Rate limited - slow down / retry later")
except openai.APIStatusError as e:
    print("OpenAI API error status code:", e.status_code)
except openai.APIConnectionError:
    print("OpenAI Error: Network problem")


try:
    resp_gemini = gem.interactions.create(model=GEMINI_MODEL, input="hi")
    print("Gemini:", resp_gemini.output_text)
except errors.APIError as e:
    print(f"Gemini API error (Status {e.code}): {e.message}")
except Exception as e:
    print("Gemini generic or network problem:", str(e))


OpenAI: Hello! How can I assist you today?
Gemini: Hello! How can I help you today?


In [31]:
r_oai = oai.responses.create(model=OPENAI_MODEL, input=MESSAGE)
print("OpenAI   :", r_oai.usage.input_tokens, r_oai.usage.output_tokens)

r_gem = gem.interactions.create(model=GEMINI_MODEL, input=MESSAGE)

usage = r_gem.usage
print("Gemini   :", usage.total_input_tokens, usage.total_output_tokens)

OpenAI   : 25 174
Gemini   : 21 520


In [2]:
def cost(in_tok, out_tok, in_price_per_mtok, out_price_per_mtok):
    return in_tok / 1e6 * in_price_per_mtok + out_tok / 1e6 * out_price_per_mtok

print("$", cost(150, 300, 2.50, 10.00))

$ 0.0033749999999999995
